# Phase 3 — Embedding extraction (external GENIE BPC cohorts)

This notebook computes **MedGemma-1.5-4B** embeddings for the external
**AACR GENIE BPC** cohorts used in the frozen-model validation (the MSK-fit
featurizer and survival head are applied without retraining; only the external
embeddings are new). MedGemma is the extractor carried into Phase 3 because it is
the only one that adds signal in Phase 1.

The model is loaded **once** and looped over the 5 solid tumors, so its 8.6 GB
download is paid a single time. The protocol is **identical** to Phase 1
(masked-mean, `max_length=512`, bf16, template `ctx_v1`) — that is what makes the
frozen MSK model applicable. The only differences vs Phase 1: one input parquet
per cohort and a `genie_<cohort>` prefix on the output so it never collides with
the MSK embeddings.

## Prerequisites

1. Build the external prompts locally (one per tumor) and upload them to
   `PROMPTS_DIR`:
   ```bash
   for c in PANC NSCLC CRC BrCa Prostate; do
       python scripts/40_build_external.py --cohort "$c"   # -> data/interim/genie_<c>_prompts.parquet
   done
   ```
2. Accept the MedGemma license and create a read token (see the load cell).

Set `Runtime -> Change runtime type -> GPU (T4)` before running.

In [ ]:
# Colab: install the dependencies for the forward pass.
!pip install -q -U transformers accelerate huggingface_hub pandas pyarrow

In [ ]:
# Optional: mount Google Drive so inputs/outputs persist across runtime restarts.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import numpy as np
import pandas as pd

# Extraction protocol — keep these IDENTICAL across every extractor and cohort
# so all embedding sets are apples-to-apples (same as the local CPU runs).
MAX_LENGTH = 512    # token cap; matches config/models.yaml defaults
BATCH_SIZE = 32     # lower for large models (e.g. 8 for MedGemma on a T4)
POOLING    = "mean" # masked mean over real tokens
TEMPLATE   = "ctx_v1"  # prompt template id; part of the output filename

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def masked_mean(last_hidden, attention_mask):
    """Mean over real (non-pad) tokens -> one vector per sequence."""
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


def save_by_patient(mat, patient_ids, fname):
    """Write the {PATIENT_ID, e0..e{d-1}} parquet that scripts/22_cox_grid.py and
    scripts/41_external_validate.py expect."""
    out = pd.DataFrame(mat, columns=[f"e{i}" for i in range(mat.shape[1])])
    out.insert(0, "PATIENT_ID", np.asarray(patient_ids))
    out.to_parquet(fname, index=False)
    print("wrote", fname, "->", out.shape)
    return out

# Where the genie_<cohort>_prompts.parquet inputs were uploaded, and where the
# embedding parquets are written.
PROMPTS_DIR = "/content/drive/MyDrive"
OUT_DIR     = "/content/drive/MyDrive"

# Lowercase cohort tags used in the input/output filenames.
COHORTS = ["panc", "nsclc", "crc", "brca", "prostate"]
SKIP_EXISTING = True   # don't recompute a cohort whose output already exists

In [ ]:
from huggingface_hub import login
from transformers import AutoModelForImageTextToText, AutoTokenizer

# MedGemma-1.5-4B is the primary extractor (the only one that beats the tabular
# baseline). Accept the license at https://huggingface.co/google/medgemma-1.5-4b-it
# and create a read token at https://huggingface.co/settings/tokens, then paste it.
login()

GEMMA_ID  = "google/medgemma-1.5-4b-it"
GEMMA_TAG = "medgemma15"   # matches config/models.yaml and the local pipeline
GEMMA_BATCH = 8            # smaller batch; the 4B model is heavier than MedCPT

gemma_tok = AutoTokenizer.from_pretrained(GEMMA_ID)
gemma_model = AutoModelForImageTextToText.from_pretrained(
    GEMMA_ID, torch_dtype=torch.bfloat16, device_map="cuda",
).eval()


def get_text_backbone(m):
    """Locate the text transformer inside the multimodal wrapper, so we pool the
    TEXT last_hidden_state (not the multimodal head)."""
    for path in ("model.language_model", "language_model", "model.model.language_model"):
        obj = m
        try:
            for p in path.split("."):
                obj = getattr(obj, p)
            return obj
        except AttributeError:
            continue
    raise AttributeError("text backbone not found; fall back to output_hidden_states=True")


gemma_text = get_text_backbone(gemma_model)


@torch.no_grad()
def embed_medgemma(texts):
    vecs = []
    for i in range(0, len(texts), GEMMA_BATCH):
        enc = gemma_tok(texts[i:i + GEMMA_BATCH], return_tensors="pt",
                        padding=True, truncation=True, max_length=MAX_LENGTH).to("cuda")
        h = gemma_text(input_ids=enc["input_ids"],
                       attention_mask=enc["attention_mask"]).last_hidden_state
        vecs.append(masked_mean(h, enc["attention_mask"]).float().cpu())
    return torch.cat(vecs).numpy()

In [ ]:
def run_cohort(cohort):
    df = pd.read_parquet(os.path.join(PROMPTS_DIR, f"genie_{cohort}_prompts.parquet"))
    assert {"PATIENT_ID", "prompt", "template_id"} <= set(df.columns)
    fname = f"genie_{cohort}__{GEMMA_TAG}__{TEMPLATE}__{POOLING}__by_patient.parquet"
    dest = os.path.join(OUT_DIR, fname)
    if SKIP_EXISTING and os.path.exists(dest):
        print(f"[{cohort}] skip (exists): {fname}")
        return
    print(f"[{cohort}] embedding {len(df)} prompts ...")
    mat = embed_medgemma(df["prompt"].tolist())
    save_by_patient(mat, df["PATIENT_ID"].to_numpy(), dest)


for c in COHORTS:
    run_cohort(c)

## Next step

Copy the `genie_*__by_patient.parquet` files into `data/processed/embeddings/` on
your machine, then run the frozen-model validation (the MSK fit is computed once
and reused per cohort):

```bash
python scripts/41_external_validate.py --all
```